# 4 — Class imbalance, and a better framing

The paper works on resamples enriched to 15% fraud, because no two-cluster
method can isolate a 0.17% minority. This notebook shows what that enrichment
costs, and what to do instead.

In [ ]:
#@title Install and import
%pip install -q qiskit qiskit-aer scikit-learn matplotlib pandas scipy

import json, time, itertools
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.metrics import adjusted_rand_score, roc_auc_score
from sklearn.metrics.pairwise import rbf_kernel, laplacian_kernel

sim = AerSimulator()
RES = "../results"
print("imports ok")

In [ ]:
#@title The method, inlined (no local package needed)
# Identical to the code that produced the paper. Notebook 1 checks it
# against Qiskit gate-by-gate.

# ---- batched statevector simulator (from faststate.py) ----

def zero_state(N, n):
    psi = np.zeros((N, 2 ** n), dtype=np.complex128)
    psi[:, 0] = 1.0
    return psi


def _axis(n, q):
    """Array axis (in the (N,2,2,...,2) view) holding qubit q."""
    return n - q


def apply_1q(psi, n, q, U):
    """U has shape (N, 2, 2) -- a different 2x2 per data point.

    Written as explicit 2x2 arithmetic rather than einsum: for a batched
    single-qubit gate this is ~10x faster, because einsum cannot see that the
    contracted index has length 2.
    """
    N = psi.shape[0]
    v = psi.reshape((N,) + (2,) * n)
    ax = _axis(n, q)
    v = np.moveaxis(v, ax, -1)
    shp = v.shape
    v = v.reshape(N, -1, 2)
    a, b = v[:, :, 0], v[:, :, 1]
    u00 = U[:, 0, 0][:, None]; u01 = U[:, 0, 1][:, None]
    u10 = U[:, 1, 0][:, None]; u11 = U[:, 1, 1][:, None]
    out = np.empty_like(v)
    out[:, :, 0] = u00 * a + u01 * b
    out[:, :, 1] = u10 * a + u11 * b
    out = out.reshape(shp)
    out = np.moveaxis(out, -1, ax)
    return out.reshape(N, 2 ** n)


def _apply_2x2_real(psi, n, q, c, s_):
    """Fast path for RY: real cos/sin, one per data point."""
    N = psi.shape[0]
    v = psi.reshape((N,) + (2,) * n)
    ax = _axis(n, q)
    v = np.moveaxis(v, ax, -1)
    shp = v.shape
    v = v.reshape(N, -1, 2)
    a, b = v[:, :, 0], v[:, :, 1]
    cc = c[:, None]; ss = s_[:, None]
    out = np.empty_like(v)
    out[:, :, 0] = cc * a - ss * b
    out[:, :, 1] = ss * a + cc * b
    out = out.reshape(shp)
    out = np.moveaxis(out, -1, ax)
    return out.reshape(N, 2 ** n)


def apply_diag_1q(psi, n, q, d):
    """Diagonal single-qubit gate; d has shape (N, 2). Elementwise, so much
    cheaper than a full 2x2."""
    N = psi.shape[0]
    v = psi.reshape((N,) + (2,) * n)
    ax = _axis(n, q)
    v = np.moveaxis(v, ax, -1)
    shp = v.shape
    v = v.reshape(N, -1, 2)
    v = v * d[:, None, :]
    v = v.reshape(shp)
    v = np.moveaxis(v, -1, ax)
    return v.reshape(N, 2 ** n)


def ry(psi, n, q, theta):
    return _apply_2x2_real(psi, n, q, np.cos(theta / 2), np.sin(theta / 2))


def rz(psi, n, q, theta):
    d = np.stack([np.exp(-0.5j * theta), np.exp(0.5j * theta)], axis=-1)
    return apply_diag_1q(psi, n, q, d)


def h(psi, n, q, N):
    U = np.empty((N, 2, 2), dtype=np.complex128)
    U[:, 0, 0] = U[:, 0, 1] = U[:, 1, 0] = 1 / np.sqrt(2)
    U[:, 1, 1] = -1 / np.sqrt(2)
    return apply_1q(psi, n, q, U)


def cx(psi, n, ctrl, targ):
    N = psi.shape[0]
    v = psi.reshape((N,) + (2,) * n)
    ac, at = _axis(n, ctrl), _axis(n, targ)
    v = np.moveaxis(v, (ac, at), (-2, -1))
    v = np.ascontiguousarray(v)
    tmp = v[..., 1, 0].copy()
    v[..., 1, 0] = v[..., 1, 1]
    v[..., 1, 1] = tmp
    v = np.moveaxis(v, (-2, -1), (ac, at))
    return v.reshape(N, 2 ** n)


def cz(psi, n, a, b):
    N = psi.shape[0]
    v = psi.reshape((N,) + (2,) * n)
    aa, ab = _axis(n, a), _axis(n, b)
    v = np.ascontiguousarray(np.moveaxis(v, (aa, ab), (-2, -1)))
    v[..., 1, 1] *= -1
    v = np.moveaxis(v, (-2, -1), (aa, ab))
    return v.reshape(N, 2 ** n)


def crz_pair(psi, n, i, j, theta):
    """CX(i,j) . RZ_j(theta) . CX(i,j) -- the standard ZZ interaction.
    Equivalent to a diagonal phase exp(-i theta/2 * z_i z_j)."""
    psi = cx(psi, n, i, j)
    psi = rz(psi, n, j, theta)
    psi = cx(psi, n, i, j)
    return psi


# ---- entanglement topologies + Qiskit feature maps (from fmzoo.py) ----

# ----------------------------------------------------------------------
# entanglement topologies
# ----------------------------------------------------------------------
def pairs_for(n, topology):
    if topology == "none":
        return []
    if topology == "linear":
        return [(i, i + 1) for i in range(n - 1)]
    if topology == "circular":
        return [(i, (i + 1) % n) for i in range(n)]
    if topology == "full":
        return [(i, j) for i in range(n) for j in range(i + 1, n)]
    if topology == "alternating":                      # brickwork
        a = [(i, i + 1) for i in range(0, n - 1, 2)]
        b = [(i, i + 1) for i in range(1, n - 1, 2)]
        return a + b
    if topology == "skip":                             # longer-range, sparse
        return [(i, (i + 2) % n) for i in range(n)]
    raise ValueError(topology)


# ----------------------------------------------------------------------
# feature-map families.  x is already scaled; c is the bandwidth.
# ----------------------------------------------------------------------
def fm_angle(x, c, topology, reps):
    """Pure product-state angle encoding. No entanglement by construction."""
    n = len(x)
    qc = QuantumCircuit(n)
    for _ in range(reps):
        for i in range(n):
            qc.ry(c * x[i], i)
    return qc


def fm_angle_ent(x, c, topology, reps):
    """Angle encoding, then CX entangling layer, then a second angle layer."""
    n = len(x)
    qc = QuantumCircuit(n)
    for _ in range(reps):
        for i in range(n):
            qc.ry(c * x[i], i)
        for (i, j) in pairs_for(n, topology):
            qc.cx(i, j)
    return qc


def fm_zz(x, c, topology, reps):
    """The classic ZZFeatureMap pattern, written out explicitly."""
    n = len(x)
    qc = QuantumCircuit(n)
    for _ in range(reps):
        qc.h(range(n))
        for i in range(n):
            qc.rz(2 * c * x[i], i)
        for (i, j) in pairs_for(n, topology):
            qc.cx(i, j)
            qc.rz(2 * (np.pi - c * x[i]) * (np.pi - c * x[j]), j)
            qc.cx(i, j)
    return qc


def fm_iqp(x, c, topology, reps):
    """IQP-style: H layer, single-qubit phases, then ZZ phases = x_i * x_j."""
    n = len(x)
    qc = QuantumCircuit(n)
    for _ in range(reps):
        qc.h(range(n))
        for i in range(n):
            qc.rz(c * x[i], i)
        for (i, j) in pairs_for(n, topology):
            qc.cx(i, j)
            qc.rz(c * x[i] * x[j], j)
            qc.cx(i, j)
    return qc


def fm_reupload(x, c, topology, reps):
    """Data re-uploading: alternate encoding and entangling layers, with the
    data entering at every layer (Perez-Salinas et al. style)."""
    n = len(x)
    qc = QuantumCircuit(n)
    for r in range(reps):
        for i in range(n):
            qc.ry(c * x[i], i)
            qc.rz(c * x[i] * (r + 1), i)
        for (i, j) in pairs_for(n, topology):
            qc.cx(i, j)
    return qc


def fm_hea(x, c, topology, reps):
    """Hardware-efficient: RY-RZ rotations plus CZ entanglers."""
    n = len(x)
    qc = QuantumCircuit(n)
    for _ in range(reps):
        for i in range(n):
            qc.ry(c * x[i], i)
            qc.rz(c * x[(i + 1) % n], i)
        for (i, j) in pairs_for(n, topology):
            qc.cz(i, j)
    return qc


def fm_zy(x, c, topology, reps):
    """Mixed-basis Pauli map: Z phases and Y rotations with YY couplings."""
    n = len(x)
    qc = QuantumCircuit(n)
    for _ in range(reps):
        qc.h(range(n))
        for i in range(n):
            qc.rz(2 * c * x[i], i)
        for (i, j) in pairs_for(n, topology):
            qc.ry(np.pi / 4, i); qc.ry(np.pi / 4, j)
            qc.cx(i, j)
            qc.rz(2 * c * x[i] * c * x[j], j)
            qc.cx(i, j)
            qc.ry(-np.pi / 4, i); qc.ry(-np.pi / 4, j)
    return qc

FAMILIES = {"angle": fm_angle, "angle_ent": fm_angle_ent, "zz": fm_zz,
            "iqp": fm_iqp, "reupload": fm_reupload, "hea": fm_hea, "zy": fm_zy}


def qiskit_kernel(X, family, c=1.0, topology="linear", reps=1):
    S = np.array([Statevector.from_instruction(
        FAMILIES[family](x, c, topology, reps)).data for x in X])
    return np.abs(S.conj() @ S.T) ** 2


# ---- batched versions of the same maps (from fmzoo_fast.py) ----

def _angle(X, c, topology, reps, entangle=False):
    N, n = X.shape
    psi = zero_state(N, n)
    P = pairs_for(n, topology) if entangle else []
    for _ in range(reps):
        for i in range(n):
            psi = ry(psi, n, i, c * X[:, i])
        for (i, j) in P:
            psi = cx(psi, n, i, j)
    return psi


def bfm_angle(X, c, topology, reps):
    return _angle(X, c, topology, reps, entangle=False)


def bfm_angle_ent(X, c, topology, reps):
    return _angle(X, c, topology, reps, entangle=True)


def bfm_zz(X, c, topology, reps):
    N, n = X.shape
    psi = zero_state(N, n)
    P = pairs_for(n, topology)
    for _ in range(reps):
        for i in range(n):
            psi = h(psi, n, i, N)
        for i in range(n):
            psi = rz(psi, n, i, 2 * c * X[:, i])
        for (i, j) in P:
            psi = crz_pair(psi, n, i, j,
                              2 * (np.pi - c * X[:, i]) * (np.pi - c * X[:, j]))
    return psi


def bfm_iqp(X, c, topology, reps):
    N, n = X.shape
    psi = zero_state(N, n)
    P = pairs_for(n, topology)
    for _ in range(reps):
        for i in range(n):
            psi = h(psi, n, i, N)
        for i in range(n):
            psi = rz(psi, n, i, c * X[:, i])
        for (i, j) in P:
            psi = crz_pair(psi, n, i, j, c * X[:, i] * X[:, j])
    return psi


def bfm_reupload(X, c, topology, reps):
    N, n = X.shape
    psi = zero_state(N, n)
    P = pairs_for(n, topology)
    for r in range(reps):
        for i in range(n):
            psi = ry(psi, n, i, c * X[:, i])
            psi = rz(psi, n, i, c * X[:, i] * (r + 1))
        for (i, j) in P:
            psi = cx(psi, n, i, j)
    return psi


def bfm_hea(X, c, topology, reps):
    N, n = X.shape
    psi = zero_state(N, n)
    P = pairs_for(n, topology)
    for _ in range(reps):
        for i in range(n):
            psi = ry(psi, n, i, c * X[:, i])
            psi = rz(psi, n, i, c * X[:, (i + 1) % n])
        for (i, j) in P:
            psi = cz(psi, n, i, j)
    return psi


def bfm_zy(X, c, topology, reps):
    N, n = X.shape
    psi = zero_state(N, n)
    P = pairs_for(n, topology)
    ones = np.full(N, np.pi / 4)
    for _ in range(reps):
        for i in range(n):
            psi = h(psi, n, i, N)
        for i in range(n):
            psi = rz(psi, n, i, 2 * c * X[:, i])
        for (i, j) in P:
            psi = ry(psi, n, i, ones); psi = ry(psi, n, j, ones)
            psi = crz_pair(psi, n, i, j, 2 * c * X[:, i] * c * X[:, j])
            psi = ry(psi, n, i, -ones); psi = ry(psi, n, j, -ones)
    return psi

BFAMILIES = {"angle": bfm_angle, "angle_ent": bfm_angle_ent, "zz": bfm_zz,
             "iqp": bfm_iqp, "reupload": bfm_reupload, "hea": bfm_hea, "zy": bfm_zy}


def kernel(X, family, c=1.0, topology="linear", reps=1):
    """Fidelity kernel K(x,y) = |<psi_x|psi_y>|^2, batched over all points."""
    S = BFAMILIES[family](X, c, topology, reps)
    return np.abs(S.conj() @ S.T) ** 2


# ---- kernel k-means (from qkmeans_kernel.py) ----

def kernel_kmeans(K, k=2, iters=60, seed=0, n_init=10):
    n = len(K)
    best = None
    for run in range(n_init):
        rng = np.random.default_rng(seed + 1000 * run)
        labels = rng.integers(0, k, n)
        for _ in range(iters):
            D = np.zeros((n, k))
            for j in range(k):
                m = labels == j
                if m.sum() == 0:
                    D[:, j] = np.inf; continue
                D[:, j] = (np.diag(K)
                           - 2 * K[:, m].mean(axis=1)
                           + K[np.ix_(m, m)].mean())
            new = D.argmin(axis=1)
            if (new == labels).all():
                break
            labels = new
        inertia = D[np.arange(n), labels].sum()
        if best is None or inertia < best[1]:
            best = (labels.copy(), inertia)
    return best[0]

print("core loaded: kernel(), kernel_kmeans(), FAMILIES")

In [ ]:
#@title Load the ULB credit-card data (downloads from OpenML, ~5 s)
full = fetch_openml(data_id=1597, as_frame=True, parser="auto").frame
full["Class"] = full["Class"].astype(int)
FEAT = json.load(open(f"{RES}/features.json"))["order"]

def draw(seed, nq, n_fraud=60, n_legit=340):
    """The exact resampling used for every number in the paper."""
    f = full[full.Class == 1].sample(n_fraud, random_state=seed)
    l = full[full.Class == 0].sample(n_legit, random_state=seed)
    d = pd.concat([f, l]).sample(frac=1, random_state=seed)
    X = StandardScaler().fit_transform(d[FEAT[:nq]].values)
    return MinMaxScaler((0, np.pi)).fit_transform(X), d["Class"].values

print(f"{len(full):,} transactions, {(full.Class==1).sum()} frauds "
      f"({100*(full.Class==1).mean():.3f}%)")
print("feature order:", FEAT[:12])

In [ ]:
#@title What happens as fraud becomes rare (stored sweep)
Bd = json.load(open(f"{RES}/baserate.json"))
df = pd.DataFrame(Bd["sweep"])[["pct", "q_ari", "c_ari", "km_ari",
                                "q_prec", "km_prec", "q_auc", "c_auc", "dist_auc"]]
df.columns = ["fraud %", "ARI quantum", "ARI classical", "ARI k-means",
              "precision quantum", "precision k-means",
              "AUROC quantum", "AUROC classical", "AUROC centroid dist."]
display(df.round(3).set_index("fraud %"))

Read it in two halves.

**Clustering (ARI columns).** Both kernel methods collapse below about 5%
fraud, while plain $k$-means degrades gracefully. The precision columns show
why: a kernel with one global bandwidth cannot wrap a tight cluster around a
few extreme outliers, so it absorbs ordinary points instead.

**Anomaly scoring (AUROC columns).** Score each point by its mean kernel
similarity to all others and flag the least similar. Same Gram matrix, no
clustering. This survives the imbalance — but the quantum kernel only matches
a classical kernel, and both are matched by plain distance to the centroid.

In [ ]:
#@title Verify the anomaly score yourself, at realistic imbalance
NQ = 12
cfg = json.load(open(f"{RES}/scaling.json"))["by_n"][str(NQ)]["best_quantum"]
print("quantum map:", cfg, "\n")

def anomaly_scores(K):
    n = len(K)
    return (K.sum(axis=1) - np.diag(K)) / (n - 1)      # low = anomalous

for nf, nl in [(20, 380), (4, 396)]:
    X, y = draw(0, NQ, nf, nl)
    Kq = kernel(X, cfg["family"], cfg["bandwidth"], cfg["topology"], cfg["reps"])
    Kc = laplacian_kernel(X, gamma=0.165)
    dist = np.linalg.norm(X - X.mean(axis=0), axis=1)
    print(f"{100*nf/(nf+nl):5.1f}% fraud   "
          f"AUROC quantum {roc_auc_score(y, -anomaly_scores(Kq)):.3f}   "
          f"classical {roc_auc_score(y, -anomaly_scores(Kc)):.3f}   "
          f"centroid distance {roc_auc_score(y, dist):.3f}")

In [ ]:
#@title At the true base rate (stored result)
for r in Bd["true_rate"]:
    print(f"{r['pct']:.2f}% fraud, N={r['n_fraud']+r['n_legit']}:  "
          f"AUROC quantum {r['q_auc']:.3f}  classical {r['c_auc']:.3f}  "
          f"centroid {r['dist_auc']:.3f}")
print("\nThe reframing helps a great deal. The quantum kernel does not.")